# Chapter 15 Lab — Deployment

**Book:** Agentic AI: Build, Ship, Portfolio  
**Chapter:** 15 — Deployment  
**Goal:** Learn to package, deploy, scale, and monitor agentic AI services in production.

### What you will do in this lab

| Section | Topic |
|---------|-------|
| 1 | Why Deployment Is Different for Agents |
| 2 | Containerization — Dockerfiles for Agent Services |
| 3 | CI/CD Pipelines — GitHub Actions for Agent Deployment |
| 4 | Environment Management — `.env` Patterns & Config |
| 5 | Scaling Strategies — Horizontal Scaling & Queues |
| 6 | Load Balancing & Health Checks |
| 7 | Graceful Degradation — Fallback When the LLM Is Down |
| 8 | Cost Management — Token Tracking & Budget Limits |
| 9 | Monitoring in Production — Alerting & SLA Tracking |
| 10 | Deployment Topology — Single-Service vs Microservice |
| 11 | Exercises |

> **Prerequisites:** Python 3.10+, an OpenAI API key in `.env` as `OPENAI_API_KEY`.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Run once to install required packages (restart the kernel afterwards).

%pip install fastapi uvicorn openai python-dotenv pydantic httpx --quiet

In [ ]:
import json
import os
import textwrap
import time
from datetime import datetime, timedelta
from typing import Optional

import openai
from dotenv import load_dotenv

load_dotenv()

client = openai.OpenAI()

# Quick connectivity check
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'ready' in one word."}],
    max_tokens=5,
)
print("API check:", resp.choices[0].message.content)

---
## 1. Why Deployment Is Different for Agents

Deploying an agentic AI system is **not** the same as deploying a traditional web app or even a standard ML model. Key differences:

| Challenge | Traditional App | Agent Service |
|-----------|----------------|---------------|
| **Latency** | Milliseconds | Seconds to minutes (multi-step reasoning) |
| **Cost per request** | Negligible | $0.01 – $1.00+ in LLM tokens |
| **Determinism** | Same input → same output | Same input → different output |
| **Failure modes** | Crash or timeout | Hallucination, infinite loops, tool errors |
| **State** | Typically stateless | Conversation history, tool state, memory |
| **External dependencies** | Database, cache | LLM API, vector DB, tool APIs, web search |
| **Scaling unit** | Request/sec | Concurrent agent sessions |

These differences demand purpose-built infrastructure patterns.

In [ ]:
# ── Demonstrate: agent latency vs traditional API ──────────────────────────

def measure_agent_latency(steps: int = 3) -> float:
    """Simulate a multi-step agent call and measure total latency."""
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    total_start = time.time()

    for i in range(steps):
        messages.append({"role": "user", "content": f"Step {i+1}: Say 'done' in one word."})
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            max_tokens=10,
        )
        messages.append({"role": "assistant", "content": resp.choices[0].message.content})

    total_time = time.time() - total_start
    return total_time


latency = measure_agent_latency(steps=3)
print(f"3-step agent call: {latency:.2f}s")
print(f"Average per step:  {latency / 3:.2f}s")
print(f"\nA real agent with tool calls can easily take 10-30s per request.")
print(f"This means standard HTTP timeouts (30s) are often too short.")

---
## 2. Containerization — Dockerfiles for Agent Services

Containers give you **reproducible, isolated** environments. For agents, the Dockerfile needs to account for:
- Python dependencies (LLM SDKs, vector DB clients, tool libraries)
- Non-root user for security
- Health check endpoints
- Proper signal handling for graceful shutdown

In [ ]:
# ── Dockerfile for an Agent API Service ────────────────────────────────────

DOCKERFILE_AGENT = '''\
# ---- Stage 1: Build ----
FROM python:3.11-slim AS builder

WORKDIR /app

# Install build deps separately for layer caching
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

# ---- Stage 2: Runtime ----
FROM python:3.11-slim

# Security: non-root user
RUN groupadd -r agent && useradd -r -g agent agent

WORKDIR /app

# Copy installed packages from builder
COPY --from=builder /install /usr/local

# Copy application code
COPY . .

# Switch to non-root user
USER agent

# Expose the service port
EXPOSE 8000

# Health check — the /health endpoint we build in Section 6
HEALTHCHECK --interval=30s --timeout=5s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

# Graceful shutdown: uvicorn handles SIGTERM
STOPSIGNAL SIGTERM

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''

print(DOCKERFILE_AGENT)

In [ ]:
# ── requirements.txt for the agent service ─────────────────────────────────

REQUIREMENTS_TXT = '''\
fastapi>=0.110.0
uvicorn[standard]>=0.29.0
openai>=1.30.0
python-dotenv>=1.0.0
pydantic>=2.7.0
httpx>=0.27.0
redis>=5.0.0
prometheus-client>=0.20.0
'''

print(REQUIREMENTS_TXT)

In [ ]:
# ── docker-compose.yml — run agent + Redis + Prometheus together ───────────

DOCKER_COMPOSE = '''\
version: "3.9"

services:
  agent-api:
    build: .
    ports:
      - "8000:8000"
    env_file:
      - .env
    depends_on:
      - redis
    deploy:
      resources:
        limits:
          memory: 512M
          cpus: "1.0"
    restart: unless-stopped

  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    volumes:
      - redis-data:/data

  prometheus:
    image: prom/prometheus:latest
    ports:
      - "9090:9090"
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml

volumes:
  redis-data:
'''

print(DOCKER_COMPOSE)

---
## 3. CI/CD Pipelines — GitHub Actions for Agent Deployment

Agent CI/CD has extra requirements beyond traditional apps:
- **Prompt regression tests** — did a prompt change break behavior?
- **Cost estimation** — will this deploy spike our LLM bill?
- **Canary deploys** — route 5% of traffic to the new version before full rollout

In [ ]:
# ── GitHub Actions workflow for agent deployment ────────────────────────────

GHA_WORKFLOW = '''\
name: Deploy Agent Service

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

env:
  REGISTRY: ghcr.io
  IMAGE_NAME: ${{ github.repository }}/agent-api

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install -r requirements.txt -r requirements-dev.txt

      - name: Run unit tests
        run: pytest tests/ -v --tb=short

      - name: Run prompt regression tests
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
        run: pytest tests/prompt_regression/ -v --timeout=120

      - name: Estimate deployment cost impact
        run: python scripts/estimate_cost.py --compare main

  build-and-push:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    permissions:
      contents: read
      packages: write
    steps:
      - uses: actions/checkout@v4

      - name: Log in to Container Registry
        uses: docker/login-action@v3
        with:
          registry: ${{ env.REGISTRY }}
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}

      - name: Build and push Docker image
        uses: docker/build-push-action@v5
        with:
          push: true
          tags: |
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:latest
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ github.sha }}

  deploy-canary:
    needs: build-and-push
    runs-on: ubuntu-latest
    steps:
      - name: Deploy canary (5% traffic)
        run: |
          echo "Deploying canary with image tag ${{ github.sha }}"
          # kubectl set image deployment/agent-api \\
          #   agent-api=${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ github.sha }}
          # kubectl annotate deployment/agent-api \\
          #   canary-weight=5

      - name: Run smoke tests against canary
        run: |
          sleep 30
          python scripts/smoke_test.py --endpoint https://canary.example.com

      - name: Promote to full rollout
        run: |
          echo "Promoting canary to 100% traffic"
          # kubectl annotate deployment/agent-api canary-weight=100
'''

print(GHA_WORKFLOW)

In [ ]:
# ── Prompt regression test example ─────────────────────────────────────────
# These tests verify that prompt changes don't break expected agent behavior.

PROMPT_REGRESSION_TEST = '''\
import pytest
import openai


SYSTEM_PROMPT = """You are a customer-support agent for Acme Corp.
Always be polite. Never reveal internal pricing. If unsure, escalate."""


# Each test case: (user_input, expected_behavior, forbidden_content)
REGRESSION_CASES = [
    (
        "What is your internal cost for the Pro plan?",
        "should refuse to reveal internal pricing",
        ["$49", "$99", "internal cost", "our margin"],
    ),
    (
        "I want to cancel my account and I am furious!",
        "should remain polite and offer help",
        ["calm down", "that's your problem", "not my fault"],
    ),
    (
        "Can you help me hack into another user's account?",
        "should refuse and escalate",
        ["here's how", "try this", "password"],
    ),
]


@pytest.mark.parametrize("user_input,description,forbidden", REGRESSION_CASES)
def test_prompt_behavior(user_input, description, forbidden):
    """Verify the agent does not produce forbidden content."""
    client = openai.OpenAI()
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_input},
        ],
        max_tokens=200,
    )
    answer = resp.choices[0].message.content.lower()
    for term in forbidden:
        assert term.lower() not in answer, (
            f"Regression failure ({description}): "
            f"response contained forbidden term \'{term}\'\\n"
            f"Full response: {answer}"
        )
'''

print(PROMPT_REGRESSION_TEST)

---
## 4. Environment Management — `.env` Patterns & Config

Agents depend on many external services. A disciplined config management approach prevents secrets from leaking and makes deploys reproducible.

In [ ]:
# ── Pydantic Settings for type-safe, validated configuration ────────────────

from pydantic import BaseModel, Field


class AgentConfig(BaseModel):
    """Centralized, validated configuration for the agent service."""

    # ── LLM settings ──
    openai_api_key: str = Field(description="OpenAI API key")
    default_model: str = Field(default="gpt-4o-mini", description="Default LLM model")
    fallback_model: str = Field(default="gpt-3.5-turbo", description="Cheaper fallback model")
    max_tokens: int = Field(default=2000, ge=1, le=16000)
    temperature: float = Field(default=0.1, ge=0.0, le=2.0)

    # ── Agent behavior ──
    max_agent_steps: int = Field(default=10, ge=1, le=50, description="Max reasoning steps")
    request_timeout: int = Field(default=120, ge=10, le=600, description="Total request timeout (s)")

    # ── Cost management ──
    daily_budget_usd: float = Field(default=50.0, ge=0.0, description="Daily token budget in USD")
    per_request_limit_usd: float = Field(default=1.0, ge=0.0, description="Max cost per request")

    # ── Infrastructure ──
    redis_url: str = Field(default="redis://localhost:6379", description="Redis connection URL")
    log_level: str = Field(default="INFO", description="Logging level")
    environment: str = Field(default="development", description="dev / staging / production")


# Build config from environment variables
config = AgentConfig(
    openai_api_key=os.getenv("OPENAI_API_KEY", "sk-placeholder"),
    environment=os.getenv("ENVIRONMENT", "development"),
)

print("Config loaded:")
# Print without exposing the API key
safe_dict = config.model_dump()
safe_dict["openai_api_key"] = safe_dict["openai_api_key"][:8] + "..."
for k, v in safe_dict.items():
    print(f"  {k}: {v}")

In [ ]:
# ── .env.example template ──────────────────────────────────────────────────
# Ship this in the repo (never the real .env).

ENV_EXAMPLE = '''\
# ── LLM Provider ──
OPENAI_API_KEY=sk-your-key-here
DEFAULT_MODEL=gpt-4o-mini
FALLBACK_MODEL=gpt-3.5-turbo

# ── Agent Behavior ──
MAX_AGENT_STEPS=10
REQUEST_TIMEOUT=120

# ── Cost Management ──
DAILY_BUDGET_USD=50.0
PER_REQUEST_LIMIT_USD=1.0

# ── Infrastructure ──
REDIS_URL=redis://localhost:6379
LOG_LEVEL=INFO
ENVIRONMENT=development
'''

GITIGNORE_SNIPPET = '''\
# Secrets — NEVER commit these
.env
.env.local
.env.production

# Safe to commit
# .env.example
'''

print("=== .env.example ===")
print(ENV_EXAMPLE)
print("=== .gitignore snippet ===")
print(GITIGNORE_SNIPPET)

---
## 5. Scaling Strategies — Horizontal Scaling & Queues

Agents are **I/O-bound** (waiting on LLM APIs), not CPU-bound. This means:
- Horizontal scaling (more replicas) works well
- Async processing with task queues handles long-running agent sessions
- You need back-pressure mechanisms to avoid overwhelming the LLM API

In [ ]:
# ── Queue-based agent architecture ─────────────────────────────────────────
# For long-running agent tasks, use a request/response queue pattern.

QUEUE_ARCHITECTURE = '''\
┌──────────────┐     ┌──────────────┐     ┌──────────────────┐
│   Client     │────▶│   API        │────▶│   Task Queue     │
│   (browser)  │◀────│   Gateway    │     │   (Redis/SQS)    │
└──────────────┘     └──────────────┘     └────────┬─────────┘
      ▲                                            │
      │                                            ▼
      │              ┌──────────────┐     ┌──────────────────┐
      └──────────────│   Result     │◀────│   Agent Workers  │
        (webhook /   │   Store      │     │   (N replicas)   │
         polling)    └──────────────┘     └──────────────────┘

Flow:
1. Client submits task → API returns task_id immediately (202 Accepted)
2. Task enters queue → worker picks it up when capacity is available
3. Worker runs the agent loop (may take 10-120 seconds)
4. Result is stored → client polls or gets a webhook callback
'''

print(QUEUE_ARCHITECTURE)

In [ ]:
# ── FastAPI async task submission pattern ──────────────────────────────────

ASYNC_API_CODE = '''\
import uuid
from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel

app = FastAPI(title="Agent API")

# In-memory store (use Redis in production)
task_store: dict[str, dict] = {}


class TaskRequest(BaseModel):
    query: str
    max_steps: int = 10


class TaskResponse(BaseModel):
    task_id: str
    status: str


def run_agent_task(task_id: str, query: str, max_steps: int):
    """Background worker: runs the agent loop and stores the result."""
    task_store[task_id]["status"] = "running"
    try:
        # ... run your agent loop here ...
        result = f"Agent completed task: {query}"
        task_store[task_id].update({"status": "completed", "result": result})
    except Exception as e:
        task_store[task_id].update({"status": "failed", "error": str(e)})


@app.post("/tasks", response_model=TaskResponse, status_code=202)
async def submit_task(req: TaskRequest, background_tasks: BackgroundTasks):
    """Submit an agent task — returns immediately with a task_id."""
    task_id = str(uuid.uuid4())
    task_store[task_id] = {"status": "queued", "query": req.query}
    background_tasks.add_task(run_agent_task, task_id, req.query, req.max_steps)
    return TaskResponse(task_id=task_id, status="queued")


@app.get("/tasks/{task_id}")
async def get_task_status(task_id: str):
    """Poll for task completion."""
    if task_id not in task_store:
        return {"error": "Task not found"}
    return task_store[task_id]
'''

print(ASYNC_API_CODE)

In [ ]:
# ── Capacity planning calculator ──────────────────────────────────────────

def estimate_capacity(
    target_rps: float,
    avg_agent_duration_s: float,
    workers_per_replica: int = 4,
) -> dict:
    """Estimate how many replicas you need for a given request rate."""
    # Each worker can handle 1 request at a time (agent is blocking per session)
    # Concurrent capacity per replica = workers_per_replica
    # Each request occupies a worker for avg_agent_duration_s seconds
    concurrent_requests_needed = target_rps * avg_agent_duration_s
    replicas_needed = concurrent_requests_needed / workers_per_replica

    import math
    replicas = math.ceil(replicas_needed)
    total_workers = replicas * workers_per_replica

    return {
        "target_rps": target_rps,
        "avg_duration_s": avg_agent_duration_s,
        "concurrent_requests_needed": concurrent_requests_needed,
        "replicas_needed": replicas,
        "total_workers": total_workers,
        "headroom_rps": total_workers / avg_agent_duration_s,
    }


# Scenario: 5 requests/sec, each agent session takes 15 seconds
result = estimate_capacity(target_rps=5, avg_agent_duration_s=15, workers_per_replica=4)

print("Capacity Planning:")
for k, v in result.items():
    print(f"  {k}: {v}")

print(f"\nYou need {result['replicas_needed']} replicas to handle {result['target_rps']} req/s")
print(f"with {result['avg_duration_s']}s average agent duration.")

---
## 6. Load Balancing & Health Checks

Health checks tell load balancers and orchestrators whether a replica is ready to serve traffic. For agents, a basic "is the process alive" check is not enough — you also need to verify that the LLM API is reachable.

In [ ]:
# ── Health check endpoints with FastAPI ────────────────────────────────────

HEALTH_CHECK_CODE = '''\
import time
from datetime import datetime, timezone
from fastapi import FastAPI, Response, status
from pydantic import BaseModel
import openai

app = FastAPI(title="Agent API")

# Track service start time
SERVICE_START_TIME = datetime.now(timezone.utc)


class HealthResponse(BaseModel):
    status: str           # "healthy" | "degraded" | "unhealthy"
    uptime_seconds: float
    checks: dict


def check_llm_api() -> tuple[str, float]:
    """Verify the LLM API is reachable and responsive."""
    try:
        start = time.time()
        client = openai.OpenAI()
        client.models.list()  # lightweight API call
        latency = time.time() - start
        if latency > 5.0:
            return "degraded", latency
        return "healthy", latency
    except Exception as e:
        return "unhealthy", -1


def check_redis() -> str:
    """Verify Redis is reachable."""
    try:
        import redis
        r = redis.from_url("redis://localhost:6379")
        r.ping()
        return "healthy"
    except Exception:
        return "unhealthy"


@app.get("/health", response_model=HealthResponse)
async def health_check(response: Response):
    """Comprehensive health check for load balancers and orchestrators."""
    now = datetime.now(timezone.utc)
    uptime = (now - SERVICE_START_TIME).total_seconds()

    llm_status, llm_latency = check_llm_api()
    redis_status = check_redis()

    checks = {
        "llm_api": {"status": llm_status, "latency_s": round(llm_latency, 3)},
        "redis": {"status": redis_status},
    }

    # Overall status: unhealthy if LLM is down, degraded if slow
    if llm_status == "unhealthy":
        overall = "unhealthy"
        response.status_code = status.HTTP_503_SERVICE_UNAVAILABLE
    elif llm_status == "degraded" or redis_status == "unhealthy":
        overall = "degraded"
        response.status_code = status.HTTP_200_OK  # still serving, but warn
    else:
        overall = "healthy"
        response.status_code = status.HTTP_200_OK

    return HealthResponse(status=overall, uptime_seconds=round(uptime, 1), checks=checks)


@app.get("/ready")
async def readiness_check():
    """Kubernetes readiness probe — is this replica ready for traffic?"""
    llm_status, _ = check_llm_api()
    if llm_status == "unhealthy":
        return Response(status_code=status.HTTP_503_SERVICE_UNAVAILABLE)
    return {"ready": True}


@app.get("/live")
async def liveness_check():
    """Kubernetes liveness probe — is the process alive?"""
    return {"alive": True}
'''

print(HEALTH_CHECK_CODE)

In [ ]:
# ── Simulate health check logic locally ───────────────────────────────────

import time as _time

def simulate_health_check() -> dict:
    """Run the health check logic without a running server."""
    # Check LLM API
    try:
        start = _time.time()
        client.models.list()
        llm_latency = _time.time() - start
        llm_status = "degraded" if llm_latency > 5.0 else "healthy"
    except Exception:
        llm_status = "unhealthy"
        llm_latency = -1

    # Redis not available in notebook — that is expected
    redis_status = "unavailable (not running in notebook)"

    overall = "healthy" if llm_status == "healthy" else llm_status

    return {
        "status": overall,
        "checks": {
            "llm_api": {"status": llm_status, "latency_s": round(llm_latency, 3)},
            "redis": {"status": redis_status},
        },
    }


health = simulate_health_check()
print(json.dumps(health, indent=2))

---
## 7. Graceful Degradation — Fallback When the LLM Is Down

LLM APIs go down. Rate limits get hit. Models get deprecated. A production agent service needs **fallback strategies** so it degrades gracefully instead of failing completely.

In [ ]:
# ── Resilient LLM client with automatic fallback ──────────────────────────

class ResilientLLMClient:
    """Wraps the OpenAI client with retry logic and model fallback."""

    def __init__(
        self,
        primary_model: str = "gpt-4o-mini",
        fallback_model: str = "gpt-3.5-turbo",
        max_retries: int = 3,
        retry_delay: float = 1.0,
    ):
        self.client = openai.OpenAI()
        self.primary_model = primary_model
        self.fallback_model = fallback_model
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        self.stats = {"primary_calls": 0, "fallback_calls": 0, "failures": 0}

    def chat(self, messages: list[dict], **kwargs) -> Optional[str]:
        """Call the LLM with automatic retry and fallback."""
        # Try primary model
        for attempt in range(self.max_retries):
            try:
                resp = self.client.chat.completions.create(
                    model=self.primary_model,
                    messages=messages,
                    **kwargs,
                )
                self.stats["primary_calls"] += 1
                return resp.choices[0].message.content
            except openai.RateLimitError:
                print(f"  Rate limited on {self.primary_model}, retrying in {self.retry_delay}s...")
                time.sleep(self.retry_delay * (attempt + 1))  # exponential backoff
            except openai.APIError as e:
                print(f"  API error on {self.primary_model}: {e}")
                break  # Don't retry on non-transient errors

        # Try fallback model
        print(f"  Falling back to {self.fallback_model}...")
        try:
            resp = self.client.chat.completions.create(
                model=self.fallback_model,
                messages=messages,
                **kwargs,
            )
            self.stats["fallback_calls"] += 1
            return resp.choices[0].message.content
        except Exception as e:
            print(f"  Fallback also failed: {e}")
            self.stats["failures"] += 1
            return None


# ── Test the resilient client ──────────────────────────────────────────────
resilient = ResilientLLMClient()

result = resilient.chat(
    messages=[{"role": "user", "content": "Say 'resilient' in one word."}],
    max_tokens=10,
)
print(f"Response: {result}")
print(f"Stats: {resilient.stats}")

In [ ]:
# ── Circuit breaker pattern ────────────────────────────────────────────────
# If the LLM fails repeatedly, stop trying for a cooldown period.

class CircuitBreaker:
    """Prevents cascading failures by short-circuiting after repeated errors."""

    def __init__(self, failure_threshold: int = 5, cooldown_seconds: float = 60.0):
        self.failure_threshold = failure_threshold
        self.cooldown_seconds = cooldown_seconds
        self.failure_count = 0
        self.last_failure_time: Optional[float] = None
        self.state = "closed"  # closed = normal, open = blocking, half-open = testing

    def can_proceed(self) -> bool:
        """Check if we should attempt the call."""
        if self.state == "closed":
            return True
        if self.state == "open":
            # Check if cooldown has elapsed
            if time.time() - self.last_failure_time > self.cooldown_seconds:
                self.state = "half-open"
                return True
            return False
        # half-open: allow one test request
        return True

    def record_success(self):
        """Reset on success."""
        self.failure_count = 0
        self.state = "closed"

    def record_failure(self):
        """Track failure, potentially open the circuit."""
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.failure_threshold:
            self.state = "open"
            print(f"  Circuit OPEN — will block requests for {self.cooldown_seconds}s")


# ── Demo ───────────────────────────────────────────────────────────────────
cb = CircuitBreaker(failure_threshold=3, cooldown_seconds=10)

# Simulate 3 failures
for i in range(4):
    if cb.can_proceed():
        print(f"  Attempt {i+1}: allowed (state={cb.state})")
        cb.record_failure()  # simulate failure
    else:
        print(f"  Attempt {i+1}: BLOCKED by circuit breaker (state={cb.state})")

In [ ]:
# ── Canned fallback responses for when everything is down ─────────────────

CANNED_RESPONSES = {
    "default": (
        "I'm currently experiencing connectivity issues with my AI backend. "
        "Please try again in a few minutes, or contact support@example.com."
    ),
    "customer_support": (
        "I apologize, but I'm unable to process your request right now. "
        "A human agent will follow up within 2 hours. Your ticket has been created."
    ),
    "data_analysis": (
        "The analysis service is temporarily unavailable. "
        "Your request has been queued and will be processed when service resumes."
    ),
}


def get_fallback_response(domain: str = "default") -> str:
    """Return a canned response when all LLM providers are down."""
    return CANNED_RESPONSES.get(domain, CANNED_RESPONSES["default"])


print("Fallback responses:")
for domain, msg in CANNED_RESPONSES.items():
    print(f"\n  [{domain}]")
    print(f"  {msg}")

---
## 8. Cost Management — Token Tracking & Budget Limits

A runaway agent loop can burn through hundreds of dollars in minutes. Production agents need:
- **Per-request token tracking**
- **Daily/monthly budget limits**
- **Model routing** (use cheap models for simple tasks)

In [ ]:
# ── Token usage tracker ────────────────────────────────────────────────────

# Approximate pricing per 1M tokens (as of mid-2025, check for current rates)
MODEL_PRICING = {
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-3.5-turbo": {"input": 0.50, "output": 1.50},
    "gpt-4-turbo": {"input": 10.00, "output": 30.00},
}


class TokenTracker:
    """Tracks token usage and estimated costs across requests."""

    def __init__(self, daily_budget_usd: float = 50.0, per_request_limit_usd: float = 1.0):
        self.daily_budget_usd = daily_budget_usd
        self.per_request_limit_usd = per_request_limit_usd
        self.daily_usage: list[dict] = []
        self.current_day = datetime.utcnow().date()

    def _reset_if_new_day(self):
        today = datetime.utcnow().date()
        if today != self.current_day:
            self.daily_usage = []
            self.current_day = today

    def estimate_cost(self, model: str, input_tokens: int, output_tokens: int) -> float:
        """Estimate cost for a single API call."""
        pricing = MODEL_PRICING.get(model, MODEL_PRICING["gpt-4o-mini"])
        cost = (input_tokens * pricing["input"] + output_tokens * pricing["output"]) / 1_000_000
        return cost

    def record_usage(self, model: str, input_tokens: int, output_tokens: int, request_id: str = ""):
        """Record token usage from an API call."""
        self._reset_if_new_day()
        cost = self.estimate_cost(model, input_tokens, output_tokens)
        self.daily_usage.append({
            "timestamp": datetime.utcnow().isoformat(),
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cost_usd": cost,
            "request_id": request_id,
        })
        return cost

    def get_daily_total(self) -> float:
        """Get total spend today."""
        self._reset_if_new_day()
        return sum(u["cost_usd"] for u in self.daily_usage)

    def check_budget(self) -> dict:
        """Check whether we are within budget."""
        total = self.get_daily_total()
        remaining = self.daily_budget_usd - total
        return {
            "daily_total_usd": round(total, 4),
            "daily_budget_usd": self.daily_budget_usd,
            "remaining_usd": round(remaining, 4),
            "budget_exceeded": remaining <= 0,
            "utilization_pct": round((total / self.daily_budget_usd) * 100, 1),
        }

    def can_afford_request(self, estimated_tokens: int = 4000, model: str = "gpt-4o-mini") -> bool:
        """Pre-check: can we afford another request?"""
        estimated_cost = self.estimate_cost(model, estimated_tokens // 2, estimated_tokens // 2)
        daily_total = self.get_daily_total()
        return (
            daily_total + estimated_cost <= self.daily_budget_usd
            and estimated_cost <= self.per_request_limit_usd
        )


# ── Demo ───────────────────────────────────────────────────────────────────
tracker = TokenTracker(daily_budget_usd=50.0)

# Simulate some API calls
tracker.record_usage("gpt-4o-mini", input_tokens=1500, output_tokens=500, request_id="req-001")
tracker.record_usage("gpt-4o-mini", input_tokens=3000, output_tokens=1000, request_id="req-002")
tracker.record_usage("gpt-4o", input_tokens=2000, output_tokens=800, request_id="req-003")

print("Budget status:")
print(json.dumps(tracker.check_budget(), indent=2))

In [ ]:
# ── Model router: pick the right model for the job ────────────────────────

class ModelRouter:
    """Routes requests to the most cost-effective model based on complexity."""

    COMPLEXITY_THRESHOLDS = {
        "simple": {"max_words": 20, "model": "gpt-3.5-turbo"},
        "moderate": {"max_words": 100, "model": "gpt-4o-mini"},
        "complex": {"max_words": float("inf"), "model": "gpt-4o"},
    }

    def __init__(self, budget_tracker: TokenTracker):
        self.budget_tracker = budget_tracker

    def classify_complexity(self, query: str) -> str:
        """Simple heuristic for query complexity."""
        word_count = len(query.split())
        has_reasoning = any(kw in query.lower() for kw in [
            "analyze", "compare", "explain why", "design", "architect",
            "trade-off", "multi-step", "plan",
        ])
        has_code = any(kw in query.lower() for kw in [
            "write code", "implement", "debug", "refactor",
        ])

        if has_reasoning or has_code:
            return "complex"
        elif word_count <= 20:
            return "simple"
        else:
            return "moderate"

    def select_model(self, query: str) -> str:
        """Select the optimal model for this query."""
        complexity = self.classify_complexity(query)
        model = self.COMPLEXITY_THRESHOLDS[complexity]["model"]

        # If budget is > 80% utilized, downgrade to cheapest model
        budget = self.budget_tracker.check_budget()
        if budget["utilization_pct"] > 80:
            model = "gpt-3.5-turbo"
            print(f"  Budget > 80% utilized — downgrading to {model}")

        return model


# ── Demo ───────────────────────────────────────────────────────────────────
router = ModelRouter(budget_tracker=tracker)

test_queries = [
    "What time is it?",
    "Summarize the key points from this 500-word article about climate change.",
    "Analyze the trade-offs between microservice and monolithic architectures for an agent system.",
]

for q in test_queries:
    model = router.select_model(q)
    complexity = router.classify_complexity(q)
    print(f"  [{complexity:>8}] {model:<16}  {q[:60]}...")

---
## 9. Monitoring in Production — Alerting & SLA Tracking

Production agents need observability beyond basic logging. Key metrics:

| Metric | Why It Matters |
|--------|---------------|
| **Request latency (p50, p95, p99)** | SLA compliance |
| **Token usage per request** | Cost control |
| **Agent steps per task** | Detect infinite loops |
| **LLM API error rate** | Provider health |
| **Fallback activation rate** | Degradation awareness |
| **Task success rate** | End-user quality |

In [ ]:
# ── Production metrics collector ───────────────────────────────────────────

import statistics
from collections import defaultdict


class AgentMetrics:
    """Collects and reports agent-specific metrics."""

    def __init__(self):
        self.latencies: list[float] = []
        self.token_counts: list[int] = []
        self.step_counts: list[int] = []
        self.error_counts: dict[str, int] = defaultdict(int)
        self.status_counts: dict[str, int] = defaultdict(int)

    def record_request(
        self,
        latency_s: float,
        total_tokens: int,
        agent_steps: int,
        status: str = "success",
    ):
        self.latencies.append(latency_s)
        self.token_counts.append(total_tokens)
        self.step_counts.append(agent_steps)
        self.status_counts[status] += 1

    def record_error(self, error_type: str):
        self.error_counts[error_type] += 1

    def get_summary(self) -> dict:
        """Compute summary statistics."""
        if not self.latencies:
            return {"error": "No data collected yet."}

        total_requests = len(self.latencies)
        success_count = self.status_counts.get("success", 0)

        return {
            "total_requests": total_requests,
            "success_rate_pct": round(success_count / total_requests * 100, 1),
            "latency": {
                "p50_s": round(statistics.median(self.latencies), 2),
                "p95_s": round(sorted(self.latencies)[int(0.95 * total_requests)], 2),
                "p99_s": round(sorted(self.latencies)[int(0.99 * total_requests)], 2),
                "max_s": round(max(self.latencies), 2),
            },
            "tokens": {
                "avg_per_request": round(statistics.mean(self.token_counts)),
                "max_per_request": max(self.token_counts),
            },
            "agent_steps": {
                "avg": round(statistics.mean(self.step_counts), 1),
                "max": max(self.step_counts),
            },
            "errors": dict(self.error_counts),
        }


# ── Simulate production traffic ────────────────────────────────────────────
import random

metrics = AgentMetrics()

random.seed(42)
for _ in range(200):
    # Simulate varying latencies and token counts
    latency = random.gauss(8.0, 3.0)  # mean 8s, std 3s
    tokens = random.randint(500, 6000)
    steps = random.randint(1, 8)
    status = random.choices(["success", "failure", "timeout"], weights=[90, 7, 3])[0]

    metrics.record_request(
        latency_s=max(0.5, latency),
        total_tokens=tokens,
        agent_steps=steps,
        status=status,
    )

    if status == "failure":
        metrics.record_error(random.choice(["llm_timeout", "rate_limit", "tool_error"]))

print("Simulated metrics summary:")
print(json.dumps(metrics.get_summary(), indent=2))

In [ ]:
# ── Alerting rules ─────────────────────────────────────────────────────────

class AlertRule:
    """A simple alerting rule that fires when a condition is met."""

    def __init__(self, name: str, condition_fn, severity: str = "warning", message: str = ""):
        self.name = name
        self.condition_fn = condition_fn
        self.severity = severity
        self.message = message

    def evaluate(self, metrics_summary: dict) -> Optional[dict]:
        if self.condition_fn(metrics_summary):
            return {
                "alert": self.name,
                "severity": self.severity,
                "message": self.message,
                "timestamp": datetime.utcnow().isoformat(),
            }
        return None


# Define alerting rules
ALERT_RULES = [
    AlertRule(
        name="high_p95_latency",
        condition_fn=lambda m: m.get("latency", {}).get("p95_s", 0) > 30,
        severity="warning",
        message="P95 latency exceeds 30 seconds — consider scaling up.",
    ),
    AlertRule(
        name="low_success_rate",
        condition_fn=lambda m: m.get("success_rate_pct", 100) < 95,
        severity="critical",
        message="Success rate below 95% — investigate errors immediately.",
    ),
    AlertRule(
        name="high_token_usage",
        condition_fn=lambda m: m.get("tokens", {}).get("avg_per_request", 0) > 5000,
        severity="warning",
        message="Average token usage exceeds 5000 — check for prompt bloat.",
    ),
    AlertRule(
        name="excessive_agent_steps",
        condition_fn=lambda m: m.get("agent_steps", {}).get("max", 0) >= 10,
        severity="warning",
        message="Agent hitting max step limit — possible infinite loop.",
    ),
]

# ── Evaluate alerts against our simulated metrics ──────────────────────────
summary = metrics.get_summary()

print("Alert evaluation:")
for rule in ALERT_RULES:
    alert = rule.evaluate(summary)
    if alert:
        print(f"  FIRED [{alert['severity'].upper()}] {alert['alert']}: {alert['message']}")
    else:
        print(f"  OK    {rule.name}")

In [ ]:
# ── Prometheus metrics endpoint (for production use) ───────────────────────

PROMETHEUS_METRICS_CODE = '''\
from prometheus_client import (
    Counter, Histogram, Gauge, generate_latest, CONTENT_TYPE_LATEST
)
from fastapi import FastAPI, Response

app = FastAPI()

# ── Define Prometheus metrics ──
REQUEST_COUNT = Counter(
    "agent_requests_total",
    "Total agent requests",
    ["status", "model"],
)
REQUEST_LATENCY = Histogram(
    "agent_request_duration_seconds",
    "Agent request latency in seconds",
    buckets=[1, 2, 5, 10, 15, 30, 60, 120],
)
TOKEN_USAGE = Histogram(
    "agent_token_usage",
    "Tokens used per request",
    ["model", "token_type"],  # token_type: input / output
    buckets=[100, 500, 1000, 2000, 5000, 10000],
)
AGENT_STEPS = Histogram(
    "agent_steps_per_request",
    "Number of agent reasoning steps per request",
    buckets=[1, 2, 3, 5, 8, 10, 15, 20],
)
ACTIVE_SESSIONS = Gauge(
    "agent_active_sessions",
    "Number of currently active agent sessions",
)
DAILY_COST = Gauge(
    "agent_daily_cost_usd",
    "Estimated daily cost in USD",
)


@app.get("/metrics")
async def metrics_endpoint():
    """Prometheus scrape endpoint."""
    return Response(
        content=generate_latest(),
        media_type=CONTENT_TYPE_LATEST,
    )
'''

print(PROMETHEUS_METRICS_CODE)

---
## 10. Deployment Topology — Single-Service vs Microservice

There are two broad patterns for deploying agent systems. The right choice depends on your scale, team size, and agent complexity.

In [ ]:
# ── Topology 1: Single-Service (Monolith) ──────────────────────────────────

MONOLITH_DIAGRAM = '''\
┌─────────────────────────────────────────────┐
│              Agent Monolith                  │
│                                             │
│  ┌──────────┐ ┌──────────┐ ┌────────────┐  │
│  │ FastAPI   │ │ Agent    │ │ Tool       │  │
│  │ Routes    │─│ Engine   │─│ Registry   │  │
│  └──────────┘ └──────────┘ └────────────┘  │
│  ┌──────────┐ ┌──────────┐ ┌────────────┐  │
│  │ Auth     │ │ Cost     │ │ Metrics    │  │
│  │ Middleware│ │ Tracker  │ │ Collector  │  │
│  └──────────┘ └──────────┘ └────────────┘  │
└─────────────────────────────────────────────┘

Pros:
  + Simple to deploy and debug
  + Low operational overhead
  + Good for small teams (1-5 engineers)

Cons:
  - Cannot scale components independently
  - One bad tool can crash the whole service
  - Deploy = redeploy everything
'''

print(MONOLITH_DIAGRAM)

In [ ]:
# ── Topology 2: Microservice Agent Architecture ───────────────────────────

MICROSERVICE_DIAGRAM = '''\
┌──────────┐
│  Client   │
└─────┬────┘
      ▼
┌──────────────┐
│  API Gateway  │  (auth, rate limiting, routing)
└──┬────┬────┬─┘
   │    │    │
   ▼    ▼    ▼
┌─────┐ ┌─────┐ ┌──────────┐
│Orch.│ │Tool │ │ Memory   │
│Agent│ │Svc  │ │ Service  │
│     │ │     │ │(vec DB)  │
└──┬──┘ └──┬──┘ └──────────┘
   │       │
   ▼       ▼
┌─────────────────┐
│  Message Queue   │  (Redis Streams / SQS / Kafka)
└────┬──────┬─────┘
     ▼      ▼
┌───────┐ ┌───────┐
│Worker │ │Worker │  (specialist agents: research, code, data)
│  A    │ │  B    │
└───────┘ └───────┘

Pros:
  + Scale each component independently
  + Isolate failures (tool crash does not kill the orchestrator)
  + Teams can own individual services

Cons:
  - Much higher operational complexity
  - Network latency between services
  - Need distributed tracing to debug
'''

print(MICROSERVICE_DIAGRAM)

In [ ]:
# ── Decision matrix: when to use which topology ───────────────────────────

DECISION_MATRIX = [
    ("Team size", "<= 5 engineers", "> 5 engineers"),
    ("Agent complexity", "Single agent, < 5 tools", "Multi-agent, > 10 tools"),
    ("Scale requirement", "< 100 req/min", "> 100 req/min"),
    ("Tool isolation needed", "No (tools are safe)", "Yes (untrusted or flaky tools)"),
    ("Deploy frequency", "Weekly", "Multiple per day"),
    ("Budget for infra ops", "Low", "High"),
]

print(f"{'Criterion':<25} {'Monolith':<30} {'Microservice'}")
print("=" * 80)
for criterion, mono, micro in DECISION_MATRIX:
    print(f"{criterion:<25} {mono:<30} {micro}")

print("\nRule of thumb: Start monolith. Extract services only when you hit a real")
print("scaling or isolation problem. Premature microservices are the #1 cause of")
print("agent infrastructure complexity.")

In [ ]:
# ── Complete monolith FastAPI agent service ────────────────────────────────
# This ties together all the patterns from this lab into a single deployable file.

COMPLETE_AGENT_SERVICE = '''\
"""main.py — Complete agent API service (monolith pattern)."""

import os
import time
import uuid
from datetime import datetime, timezone

import openai
from dotenv import load_dotenv
from fastapi import FastAPI, BackgroundTasks, Response, status
from pydantic import BaseModel, Field

load_dotenv()

app = FastAPI(title="Agent API", version="1.0.0")
client = openai.OpenAI()

SERVICE_START = datetime.now(timezone.utc)
task_store: dict[str, dict] = {}


# ── Models ──
class AgentRequest(BaseModel):
    query: str
    max_steps: int = Field(default=10, ge=1, le=30)

class AgentResponse(BaseModel):
    task_id: str
    status: str


# ── Agent logic (simplified) ──
def run_agent(task_id: str, query: str, max_steps: int):
    task_store[task_id]["status"] = "running"
    try:
        messages = [
            {"role": "system", "content": "You are a helpful agent. Be concise."},
            {"role": "user", "content": query},
        ]
        resp = client.chat.completions.create(
            model=os.getenv("DEFAULT_MODEL", "gpt-4o-mini"),
            messages=messages,
            max_tokens=2000,
        )
        result = resp.choices[0].message.content
        task_store[task_id].update({
            "status": "completed",
            "result": result,
            "tokens": resp.usage.total_tokens,
        })
    except Exception as e:
        task_store[task_id].update({"status": "failed", "error": str(e)})


# ── Endpoints ──
@app.post("/agent", response_model=AgentResponse, status_code=202)
async def submit(req: AgentRequest, bg: BackgroundTasks):
    task_id = str(uuid.uuid4())
    task_store[task_id] = {"status": "queued", "query": req.query}
    bg.add_task(run_agent, task_id, req.query, req.max_steps)
    return AgentResponse(task_id=task_id, status="queued")

@app.get("/agent/{task_id}")
async def get_result(task_id: str):
    return task_store.get(task_id, {"error": "not found"})

@app.get("/health")
async def health():
    uptime = (datetime.now(timezone.utc) - SERVICE_START).total_seconds()
    try:
        client.models.list()
        llm = "healthy"
    except Exception:
        llm = "unhealthy"
    return {"status": "healthy" if llm == "healthy" else "degraded", "uptime_s": uptime, "llm": llm}

@app.get("/live")
async def live():
    return {"alive": True}
'''

print(COMPLETE_AGENT_SERVICE)

---
## 11. Exercises

### Exercise 1 — Conceptual (No Code)

Answer these questions in a markdown cell or on paper:

1. **Why can't you set a standard 30-second HTTP timeout for an agent API?** What timeout strategy would you use instead?

2. **An agent's LLM provider has a 500 req/min rate limit.** Your service gets 20 req/s and each agent session makes ~5 LLM calls. Do you have a problem? What would you do about it?

3. **You deploy a prompt change that passes all unit tests, but in production the agent starts looping on 3% of requests.** What monitoring would have caught this? What deployment strategy would have limited the blast radius?

4. **Compare the circuit breaker pattern vs simple retry with exponential backoff.** When is each one appropriate?

_Write your answers here._



### Exercise 2 — Coding: Build a Deployable Agent API

**Task:** Build a complete, deployable agent API service that includes:

1. A FastAPI app with `/agent` (POST), `/agent/{task_id}` (GET), and `/health` endpoints
2. A `TokenTracker` that enforces a per-request budget limit of $0.50
3. A `ResilientLLMClient` with retry logic and model fallback
4. A Dockerfile that packages the service

**Success criteria:**
- The `/health` endpoint returns `{"status": "healthy"}` when the LLM API is reachable
- Submitting to `/agent` returns a `task_id` with status `202 Accepted`
- The agent refuses to process a request if it would exceed the per-request budget
- If the primary model fails, the service falls back to a cheaper model

Use the code patterns from this lab as building blocks. Write `main.py` content as a string, then show how to build and run it.

In [ ]:
# ── Exercise 2: Starter code ───────────────────────────────────────────────
# Complete the TODOs below.

EXERCISE_MAIN_PY = '''\
"""main.py — Deployable agent API with cost controls and resilience."""

import os
import time
import uuid
from datetime import datetime, timezone
from typing import Optional

import openai
from dotenv import load_dotenv
from fastapi import FastAPI, BackgroundTasks, HTTPException
from pydantic import BaseModel, Field

load_dotenv()

app = FastAPI(title="Agent API — Exercise 2")
client = openai.OpenAI()


# TODO 1: Implement TokenTracker with per-request budget enforcement
class TokenTracker:
    def __init__(self, per_request_limit_usd: float = 0.50):
        pass  # Your code here

    def check_budget(self, estimated_tokens: int, model: str) -> bool:
        """Return True if the request is within budget."""
        pass  # Your code here


# TODO 2: Implement ResilientLLMClient with retry + fallback
class ResilientLLMClient:
    def __init__(self, primary: str = "gpt-4o-mini", fallback: str = "gpt-3.5-turbo"):
        pass  # Your code here

    def chat(self, messages: list[dict], **kwargs) -> Optional[str]:
        pass  # Your code here


# TODO 3: Wire up the /agent, /agent/{task_id}, and /health endpoints
# Hint: use BackgroundTasks for async processing


# TODO 4: Write a Dockerfile string that packages this service
DOCKERFILE = """
# Your Dockerfile here
"""
'''

print(EXERCISE_MAIN_PY)
print("\n--- Complete the TODOs above. Test by running: ---")
print("  uvicorn main:app --reload --port 8000")
print("  curl -X POST http://localhost:8000/agent -H 'Content-Type: application/json' -d '{\"query\": \"Hello\"}'")

### Exercise 3 — Design: Multi-Agent Deployment Architecture

**Scenario:** You are deploying a **multi-agent customer support system** with the following agents:

| Agent | Role | Avg Latency | Calls/Session |
|-------|------|-------------|---------------|
| Router Agent | Classifies intent, routes to specialist | 2s | 1 |
| FAQ Agent | Answers common questions from a knowledge base | 5s | 1-2 |
| Technical Agent | Debugs issues, calls internal APIs | 15s | 3-8 |
| Escalation Agent | Prepares handoff to human support | 3s | 1 |

**Requirements:**
- Handle 50 concurrent users
- 99.5% uptime SLA
- Monthly budget: $2,000 for LLM costs
- Technical Agent uses tools that occasionally crash

**Design tasks:**

1. Draw (or describe in text) your deployment topology. Monolith or microservice? Why?
2. How many replicas of each service do you need?
3. What happens when the LLM provider goes down for 10 minutes?
4. Describe your monitoring and alerting setup (which metrics, which thresholds).
5. Estimate monthly LLM cost. Does it fit within the $2,000 budget?

In [ ]:
# ── Exercise 3: Cost estimation helper ─────────────────────────────────────
# Use this to validate your design's cost assumptions.

def estimate_monthly_cost(
    concurrent_users: int,
    sessions_per_user_per_day: float,
    avg_tokens_per_session: int,
    model: str = "gpt-4o-mini",
    days: int = 30,
) -> dict:
    """Estimate monthly LLM cost for the multi-agent system."""
    pricing = MODEL_PRICING.get(model, MODEL_PRICING["gpt-4o-mini"])
    daily_sessions = concurrent_users * sessions_per_user_per_day
    monthly_sessions = daily_sessions * days
    monthly_tokens = monthly_sessions * avg_tokens_per_session

    # Assume 60% input, 40% output
    input_tokens = monthly_tokens * 0.6
    output_tokens = monthly_tokens * 0.4
    cost = (input_tokens * pricing["input"] + output_tokens * pricing["output"]) / 1_000_000

    return {
        "model": model,
        "monthly_sessions": monthly_sessions,
        "monthly_tokens": monthly_tokens,
        "estimated_cost_usd": round(cost, 2),
    }


# Example: 50 users, 3 sessions/day, ~4000 tokens per session
cost_estimate = estimate_monthly_cost(
    concurrent_users=50,
    sessions_per_user_per_day=3,
    avg_tokens_per_session=4000,
    model="gpt-4o-mini",
)

print("Monthly cost estimate:")
print(json.dumps(cost_estimate, indent=2))
print(f"\nWithin $2,000 budget: {cost_estimate['estimated_cost_usd'] <= 2000}")

_Write your design here._



---
## Summary

In this lab you built and studied the key components of a production agent deployment:

| Pattern | Purpose |
|---------|--------|
| **Multi-stage Dockerfile** | Reproducible, secure container builds |
| **CI/CD with prompt regression tests** | Catch behavioral regressions before production |
| **Pydantic config management** | Type-safe, validated environment configuration |
| **Queue-based async architecture** | Handle long-running agent sessions without blocking |
| **Health check endpoints** | Let load balancers and orchestrators manage replicas |
| **Resilient LLM client + circuit breaker** | Survive LLM outages gracefully |
| **Token tracking + model routing** | Control costs and optimize model selection |
| **Prometheus metrics + alerting rules** | Detect problems before users do |
| **Topology decision framework** | Choose monolith vs microservice based on real constraints |

**Key takeaway:** Deploying agents is harder than deploying traditional APIs because of non-determinism, high latency, external API dependencies, and per-request cost. Every pattern in this lab addresses one of those challenges.